### Retrieving the results of the training to make predictions on the test dataset and display the confusion matrix an the interpretation tools

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, DataCollatorWithPadding
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import datasets
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [3]:
labels      = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance", "No Value"]
num_labels  = len(labels)

train_dataset       = datasets.load_from_disk("../datasets/processed_train")
validation_dataset  = datasets.load_from_disk("../datasets/processed_validation")
test_dataset        = datasets.load_from_disk("../datasets/processed_test")

In [ ]:
model       = RobertaForSequenceClassification.from_pretrained("../results", num_labels = num_labels)
tokenizer   = RobertaTokenizer.from_pretrained("../results")

In [ ]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def dataset_to_dataloader(encoded_dataset, batch_size=8):
    return DataLoader(encoded_dataset, batch_size=batch_size, collate_fn=data_collator)

test_dataloader = dataset_to_dataloader(test_dataset)

all_logits = []
all_true_labels = []

for batch in test_dataloader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = torch.tensor(batch["labels"]).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs[0]
        all_logits.append(logits.cpu().numpy())
        all_true_labels.append(labels.cpu().numpy())

# Concatenate logits and true labels
all_logits = np.concatenate(all_logits, axis=0)
all_true_labels = np.concatenate(all_true_labels, axis=0)
probs = torch.sigmoid(torch.tensor(all_logits))

### Dynamic filtering of the predictions:


In [ ]:
delta   = 0.2
gap     = 0.1

def filter_pred(probs, all_true_labels, delta=0.2, gap=0.1):
    max_probs, max_indices = probs.max(dim=1)
    predicted_labels = torch.zeros_like(probs)

    for i in range(probs.shape[0]):
        sentence_probs = probs[i]
        for j in range(len(sentence_probs)):
            if sentence_probs[j] >= (max_probs[i] - gap) and sentence_probs[j] >= (max_probs[i] - delta):
                predicted_labels[i, j] = 1  # Keep this label
            else:
                predicted_labels[i, j] = 0  # Remove this label

    num_labels = all_true_labels.shape[1]
    return predicted_labels, num_labels

predicted_labels, num_labels = filter_pred(probs, all_true_labels)

### Basic thresholding filtering of the predictions

In [ ]:
threshold = 0.5
predicted_labels = (probs >= threshold).int().numpy()

num_labels = all_true_labels.shape[1]

### Custom confusion matrix

In [ ]:
def label_error_distribution(y_true, y_pred, labels):
    num_labels = len(labels)
    label_matrix = np.zeros((num_labels, num_labels))
    total_correct = 0

    for i in range(len(y_true)):
        true_labels = np.where(y_true[i] == 1)[0]
        pred_labels = np.where(y_pred[i] == 1)[0]
        
        for t in true_labels:
            for p in pred_labels:
                label_matrix[t, p] += 1  # Track true label t → predicted label p
                if t==p:
                    total_correct += 1

    # Normalize by row to show proportions
    label_matrix = label_matrix / label_matrix.sum(axis=1, keepdims=True)
    print("sum diagonal = ", total_correct)
    print("total preds = ", np.sum(y_true))
    print("'accuracy' = ", total_correct/ np.sum(y_true))

    # Heatmap of label misclassification
    fig = plt.figure(figsize=(12, 8))
    sns.heatmap(label_matrix, annot=True, fmt=".2f", cmap="Greens", xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted Labels", color="white")
    plt.ylabel("True Labels", color="white")
    plt.title("Labels Prediction", color="white")
    fig.patch.set_facecolor('none')
    plt.show()
    return label_matrix

label_matrix = label_error_distribution(all_true_labels, predicted_labels, labels)

In [ ]:
# Get the top 5 predictions

# Extract diagonal elements (correct predictions)
diagonal_scores = np.diag(label_matrix) 
label_scores = list(zip(labels, diagonal_scores))

# Sort the labels by their diagonal score
label_scores = sorted(label_scores, key=lambda x: x[1], reverse=True)

n_best = 5
top = label_scores[:n_best]
labels_for_plot = [x[0] for x in top]
scores_for_plot = [x[1] for x in top]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(y=scores_for_plot, x=labels_for_plot, color="cornflowerblue")
plt.title(f'Top {n_best} Prediction scores', color="midnightblue", fontsize=18, fontweight='bold')
plt.ylabel('Prediction Score', color='midnightblue', fontsize=18, fontweight='bold')
plt.xticks(rotation=10, color='midnightblue', fontsize=18, fontweight='bold')
plt.yticks(color='midnightblue', fontsize=18, fontweight='bold')
plt.axhline(y=0.5, color='navy', linestyle='--', linewidth=2)
fig.patch.set_facecolor('none')
ax.set_facecolor('none')
plt.tight_layout()
plt.show()

### Explainability :

In [7]:
# que mettre comme chemin pour le file des labels ???
def load_dataset(directory, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, f"{dataset}.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)

num_labels = all_true_labels.shape[1]

In [8]:
directory_train="../datasets/valueeval24/training-english"
data = load_dataset(directory_train)

In [9]:
data["Text"][:3].tolist()

['Spain’s minister pleads for ‘millions and millions’ of immigrants',
 'Madrid.',
 '“The demographic trajectories are there.']

Visualise the impact of each word for the output classes

In [10]:
l=data["Text"].tolist()

In [11]:
id2label = {i: labels[i] for i in range(len(labels))}
print(id2label)
model.config.id2label = id2label
model.config.id2label


{0: 'Self-direction: thought', 1: 'Self-direction: action', 2: 'Stimulation', 3: 'Hedonism', 4: 'Achievement', 5: 'Power: dominance', 6: 'Power: resources', 7: 'Face', 8: 'Security: personal', 9: 'Security: societal', 10: 'Tradition', 11: 'Conformity: rules', 12: 'Conformity: interpersonal', 13: 'Humility', 14: 'Benevolence: caring', 15: 'Benevolence: dependability', 16: 'Universalism: concern', 17: 'Universalism: nature', 18: 'Universalism: tolerance', 19: 'No Value'}


{0: 'Self-direction: thought',
 1: 'Self-direction: action',
 2: 'Stimulation',
 3: 'Hedonism',
 4: 'Achievement',
 5: 'Power: dominance',
 6: 'Power: resources',
 7: 'Face',
 8: 'Security: personal',
 9: 'Security: societal',
 10: 'Tradition',
 11: 'Conformity: rules',
 12: 'Conformity: interpersonal',
 13: 'Humility',
 14: 'Benevolence: caring',
 15: 'Benevolence: dependability',
 16: 'Universalism: concern',
 17: 'Universalism: nature',
 18: 'Universalism: tolerance',
 19: 'No Value'}

The Transformers_interpret explainer

In [12]:
from transformers_interpret import SequenceClassificationExplainer
cls_explainer = SequenceClassificationExplainer(model, tokenizer)

In [144]:
attributions = cls_explainer(l[3])

In [145]:
cls_explainer.visualize()

Tokenizer problem?

In [23]:
text = "j'aime l'école" 
tokens = tokenizer.tokenize(text)

print("Tokens:", tokens)

Tokens: ['j', "'", 'a', 'ime', 'Ġl', "'", 'Ã©', 'co', 'le']


In [18]:
model2       = RobertaForSequenceClassification.from_pretrained(r"C:\Users\amine\Downloads\DASCI\Projet Pro Com\exai\checkpoint-11190-20250306T093724Z-001\checkpoint-11190", num_labels = num_labels)
tokenizer2   = RobertaTokenizer.from_pretrained(r"C:\Users\amine\Downloads\DASCI\Projet Pro Com\exai\tokenizer-20250306T094027Z-001\tokenizer")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at C:\Users\amine\Downloads\DASCI\Projet Pro Com\exai\checkpoint-11190-20250306T093724Z-001\checkpoint-11190 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
model2.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

Visualise the impact on all the output classes

In [26]:
shap_values = explainer(data["Text"][:3])
shap.plots.text(shap_values)

In [7]:
!pip freeze > requirements.txt
